# KCD 2026 · 노트북 하나로 전부 실행

코드 버전 `0bba8a0`. 이 파일 하나에 필요한 코드가 모두 들어 있습니다. 첫 셀의 `MIMIC_ROOT`만 확인하고 위에서부터 끝까지 실행하세요.

기관 PC에서 이 노트북 하나만 위에서부터 끝까지 실행하면 됩니다.

| 단계 | 하는 일 | 대략 |
|---|---|---|
| 1 | 전체 MIMIC-IV로 분석 노트북 실행 (코호트·사건·세 에이전트·평가) | 가장 오래 걸림 (labevents 전체 스캔) |
| 2 | 모델 개발·평가: Agent 3 위험, 24시간 궤적, Agent 2 예후 4경로, Agent 1 원인 8개 0/1, 그룹 Shapley, 전과 후 실험 | 수십 분 |
| 3 | 공개 MIMIC-IV Demo 100명에 학습된 모델을 적용해 시연 화면 생성 (데모 환자는 학습에서 제외됨) | 수 분 |
| 4 | 돌려줄 파일 하나로 묶기: `kcd_return_package_*.zip` | 1분 |

**돌려주는 zip에 들어가는 것**: 집계 결과(`results_full.json`, 10 미만 칸 가림), 논문 표 CSV, 집계 그림 PNG, 공개 데모 환자의 시연 화면 JSON, 실행 보고서.
**들어가지 않는 것**: 학습된 모델 파일, 전체 MIMIC-IV 환자 단위 파일, 실행된 노트북 사본. 모두 이 PC의 `run_all/` 폴더에만 남습니다.

중간에 한 단계가 실패해도 다음 단계와 압축은 계속 진행하고, 실패 내용을 보고서에 적습니다. 그 zip을 그대로 보내주세요.

**다시 실행할 때**: 코드만 바뀐 새 노트북을 받았다면 `RUN_STEPS['full'] = False`로 두면 가장 오래 걸리는 1단계를 건너뛰고 이전 결과(`kcd_work/run_all`)를 재사용합니다.

In [ ]:
# ===== 설정: 필요하면 여기만 고칩니다 =====
from pathlib import Path
import os, sys, json, time, shutil, zipfile, traceback, subprocess, platform
from datetime import datetime

BASE = Path.cwd() / 'kcd_work'         # 코드와 중간 산출물 (이 PC에만 남음)
REPO = BASE / 'code'                   # 아래 셀이 코드를 여기에 풉니다
MIMIC_ROOT = Path(r'\\10.7.12.146\데이터 및 소스\MIMIC\MIMIC4-V3.1')   # hosp/ icu/ 가 있는 폴더
MAX_PATIENTS = None        # 시간이 부족하면 20000 같은 숫자 (None = 전체)
DEMO_CSV = BASE / 'mimic-iv-demo-csv'   # 없으면 3단계에서 자동 생성
DEMO_FHIR_DIR = None       # 인터넷이 막혀 있으면 미리 받은 FHIR 데모의 fhir 폴더 경로
WORK = BASE / 'run_all'    # 환자 단위 중간 산출물 (이 PC에만 남음)
RUN_STEPS = {'full': True, 'develop': True, 'demo': True, 'package': True}
# 기관 LLM (선택). 없으면 Agent 1은 규칙 기준 모델로 실행됩니다. 비밀키는 노트북에 적지 말고 환경변수로.
# os.environ['LLM_BASE_URL'] = 'https://.../v1'; os.environ['LLM_MODEL'] = '...'
REPORT = {'started_at': datetime.now().isoformat(timespec='seconds'), 'steps': {}, 'python': platform.python_version()}
print('REPO', REPO); print('MIMIC_ROOT 접근 가능:', MIMIC_ROOT.exists())
OUTBOX = Path.cwd()   # 돌려줄 zip이 이 노트북 옆에 생깁니다


In [ ]:
# ===== 코드 풀기 (수정하지 마세요) =====
import base64, io, tarfile
PAYLOAD = "..."
if REPO.exists(): shutil.rmtree(REPO)
REPO.mkdir(parents=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode='r:gz') as t:
    t.extractall(REPO)
sys.path.insert(0, str(REPO)); WORK.mkdir(parents=True, exist_ok=True)
for m in [k for k in sys.modules if k.split('.')[0] in ('kcd_liver', 'scripts')]: del sys.modules[m]
REPORT['code_version'] = '0bba8a0'
print('코드 버전 0bba8a0:', sum(1 for _ in REPO.rglob('*.py')), '개 파이썬 파일 ->', REPO)


In [ ]:
# ===== 환경 확인 =====
need = ['pandas', 'numpy', 'sklearn', 'matplotlib', 'joblib', 'nbformat', 'nbclient', 'ipykernel']
missing = []
for m in need:
    try: __import__(m)
    except ImportError: missing.append(m)
if missing:
    print('설치가 필요합니다:', missing)
    print('pip install', ' '.join('scikit-learn' if m == 'sklearn' else m for m in missing))
    raise SystemExit('위 패키지를 설치하고 커널을 재시작한 뒤 다시 실행하세요.')
import pandas as pd, numpy as np, sklearn
REPORT['versions'] = {'pandas': pd.__version__, 'numpy': np.__version__, 'scikit-learn': sklearn.__version__}
print(REPORT['versions'])

In [ ]:
# ===== 도우미: 분석 노트북을 설정만 바꿔 실행 =====
import nbformat
from nbclient import NotebookClient
MAIN_NB = REPO / 'notebooks' / 'KCD_MIMICIV_liver_agents_v2.ipynb'
ORIG_ROOT = "ROOT = Path(r'\\\\10.7.12.146\\데이터 및 소스\\MIMIC\\MIMIC4-V3.1')"

def run_main(name, root, source, max_patients=None, extra=None, env=None):
    work = WORK / name; work.mkdir(parents=True, exist_ok=True)
    nb = nbformat.read(MAIN_NB, as_version=4)
    subs = [(ORIG_ROOT, f"ROOT = Path(r'{root}')"), ("DATA_SOURCE = 'mimic-iv'", f"DATA_SOURCE = '{source}'"),
            ("MAX_PATIENTS = 3000  # smoke run", f"MAX_PATIENTS = {max_patients!r}  # smoke run")] + list((extra or {}).items())
    hit = {a: 0 for a, _ in subs}
    for c in nb.cells:
        for a, b in subs:
            if a in c.source: c.source = c.source.replace(a, b); hit[a] += 1
    miss = [a for a, n in hit.items() if n == 0]
    if miss: raise RuntimeError(f'설정 줄을 찾지 못했습니다: {miss}')
    env = {'KCD_REPO': REPO, **(env or {})}
    old = {k: os.environ.get(k) for k in (env or {})}
    os.environ.update({k: str(v) for k, v in (env or {}).items()})
    t = time.time()
    try:
        NotebookClient(nb, timeout=None, kernel_name='python3', resources={'metadata': {'path': str(work)}}).execute()
    finally:
        nbformat.write(nb, work / 'executed_PRIVATE.ipynb')   # 환자 정보가 출력될 수 있어 압축에서 제외
        for k, v in old.items():
            if v is None: os.environ.pop(k, None)
            else: os.environ[k] = v
    return work / 'kcd_liver_v2_outputs', round((time.time() - t) / 60, 1)

def step(name, fn):
    t = time.time()
    try:
        out = fn(); REPORT['steps'][name] = {'status': 'ok', 'minutes': round((time.time() - t) / 60, 1), **(out or {})}
    except BaseException as ex:
        REPORT['steps'][name] = {'status': 'failed', 'minutes': round((time.time() - t) / 60, 1), 'error': f'{type(ex).__name__}: {ex}',
                                 'trace_tail': traceback.format_exc()[-1500:]}
        print('실패:', name, ex)
    print(name, REPORT['steps'][name]['status'], REPORT['steps'][name]['minutes'], '분')

In [ ]:
# ===== 1단계: 전체 MIMIC-IV 분석 노트북 =====
FULL_OUT = WORK / 'full' / 'kcd_liver_v2_outputs'
def s1():
    out, m = run_main('full', MIMIC_ROOT, 'mimic-iv', MAX_PATIENTS)
    flow = pd.read_csv(out / '04_cohort_flow.csv').to_dict('records')
    return {'notebook_minutes': m, 'cohort_flow': flow}
if RUN_STEPS['full']: step('1_full_notebook', s1)

In [ ]:
# ===== 2단계: 모델 개발·평가 (전체 MIMIC-IV, 공개 데모 환자 제외) =====
MODELS = WORK / 'models'
def s2():
    import importlib, scripts.develop_models as dm
    importlib.reload(dm)
    dm.main(['--out-dir', str(FULL_OUT), '--mimic-root', str(MIMIC_ROOT), '--source', 'mimic-iv', '--model-dir', str(MODELS)])
    R = json.loads((FULL_OUT / 'results_full.json').read_text(encoding='utf-8'))
    return {'agent3_eval_rows': R.get('agent3_development', {}).get('test_rows'), 'models_saved_locally': sorted(p.name for p in MODELS.glob('*.joblib'))}
if RUN_STEPS['develop']: step('2_develop_models', s2)

In [ ]:
# ===== 3단계: 공개 데모 100명에 학습된 모델을 적용해 시연 화면 생성 =====
def s3():
    if not (DEMO_CSV / 'hosp' / 'labevents.csv.gz').exists():
        args = [sys.executable, str(REPO / 'scripts' / 'fhir_demo_to_csv.py'), '--out', str(DEMO_CSV)]
        if DEMO_FHIR_DIR: args += ['--fhir-dir', str(DEMO_FHIR_DIR)]
        r = subprocess.run(args, capture_output=True, text=True); print(r.stdout[-500:], r.stderr[-500:])
        if r.returncode: raise RuntimeError('데모 데이터 준비 실패: 인터넷이 막혀 있으면 DEMO_FHIR_DIR를 지정하세요')
    out, m = run_main('demo', DEMO_CSV, 'mimic-iv-demo', None,
                      extra={'BUILD_AGENT3_REFERENCE=False': 'BUILD_AGENT3_REFERENCE=True'}, env={'KCD_MODEL_DIR': MODELS})
    # supplementary figures should show the FULL cohort aggregates again (demo run overwrote summary.json)
    if (FULL_OUT / '40_site_aggregates.json').exists():
        subprocess.run([sys.executable, str(REPO / 'scripts' / 'export_site_data.py'), '--out-dir', str(FULL_OUT), '--source', 'mimic-iv', '--aggregates-only'], check=True)
    return {'notebook_minutes': m, 'demo_cases': sorted(p.name for p in (REPO / 'site' / 'data' / 'cases').glob('*.json'))}
if RUN_STEPS['demo']: step('3_demo_replay', s3)

In [ ]:
# ===== 4단계: 돌려줄 파일 하나로 묶기 (집계와 공개 데모만) =====
FORBIDDEN_KEYS = {'subject_id', 'hadm_id', 'stay_id', 'labevent_id', 'case_id'}
def scan_ids(o, path='', hits=None):
    hits = [] if hits is None else hits
    if isinstance(o, dict):
        for k, v in o.items():
            if k in FORBIDDEN_KEYS: hits.append(path + '/' + k)
            scan_ids(v, path + '/' + str(k), hits)
    elif isinstance(o, list):
        for v in o[:2000]: scan_ids(v, path, hits)
    return hits
def s4():
    site = REPO / 'site' / 'data'
    if (FULL_OUT / 'results_full.json').exists():
        shutil.copy(FULL_OUT / 'results_full.json', site / 'results_full.json')
        hits = scan_ids(json.loads((site / 'results_full.json').read_text(encoding='utf-8')))
        if hits: raise RuntimeError(f'results_full.json에 환자 식별 키가 있습니다: {hits[:5]}')
    guard = subprocess.run([sys.executable, str(REPO / 'scripts' / 'check_public_data.py')], capture_output=True, text=True, cwd=REPO)
    print(guard.stdout)
    if guard.returncode: raise RuntimeError('공개 데이터 검사 실패: ' + guard.stdout)
    stamp = datetime.now().strftime('%Y%m%d_%H%M')
    zp = OUTBOX / f'kcd_return_package_{stamp}.zip'
    REPORT['finished_at'] = datetime.now().isoformat(timespec='seconds')
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in site.rglob('*'):
            if p.is_file() and p.suffix in {'.json'}: z.write(p, 'site_data/' + str(p.relative_to(site)))
        for sub, pat in [('results_tables', '*'), ('figures', '*.png')]:
            for p in (FULL_OUT / sub).glob(pat) if (FULL_OUT / sub).exists() else []:
                z.write(p, f'full_{sub}/' + p.name)
        for name in ['04_cohort_flow.csv', '18_alert_aggregation.csv', '33_threshold_sensitivity.csv', '02_lab_quality.csv']:
            if (FULL_OUT / name).exists(): z.write(FULL_OUT / name, 'full_aggregates/' + name)
        z.writestr('run_report.json', json.dumps(REPORT, ensure_ascii=False, indent=1, default=str))
    names = zipfile.ZipFile(zp).namelist()
    bad = [n for n in names if any(s in n for s in ('_private', '.sqlite', '.joblib', 'EVALUATION_ONLY', 'executed_'))]
    if bad: zp.unlink(); raise RuntimeError(f'금지 파일이 포함되어 압축을 지웠습니다: {bad}')
    print(f'완성: {zp}  ({zp.stat().st_size/1e6:.1f} MB, 파일 {len(names)}개)')
    return {'package': str(zp), 'files': len(names)}
if RUN_STEPS['package']: step('4_package', s4)
print(json.dumps({k: v.get('status') for k, v in REPORT['steps'].items()}, ensure_ascii=False))

## 보낼 것

`kcd_return_package_*.zip` 하나를 대화에 올려주세요. 실패한 단계가 있어도 zip 안의 `run_report.json`에 이유가 적혀 있으니 그대로 보내주시면 됩니다.

학습된 모델(`run_all/models/*.joblib`)은 보내지 않습니다. 시연 화면에 필요한 모델 예측은 3단계에서 이미 데모 환자 화면에 담겼습니다.